# Persona Vectors: Preventative Steering During Training

`persona_vectors_6.ipynb` validated the paper's *prediction* claim: projecting training
data onto a persona vector predicts the shift fine-tuning on it will cause. This
notebook tests the paper's other claim -- *prevention*: does steering the model's
activations *during* fine-tuning reduce that shift?

Fine-tunes Qwen2.5-7B-Instruct on the same `dataset/evil/misaligned_2.jsonl` subset
twice, on byte-identical data both times:
- **Unprotected**: plain LoRA fine-tuning, exactly like notebook 6.
- **Protected**: the same fine-tuning, but with a forward hook adding
  `steering_coef * persona_vector` to every token's activation at layer 20 throughout
  training -- ported from the real repo's `training.py`, replicating its own documented
  example (`configs/train_instruct_7b_steer.json`: `type=steer, coeff=5.0, layer=20`)
  exactly.

**Why the coefficient is positive** (the same direction as "evil", not away from it):
forcibly injecting the trait direction during training means the model doesn't need to
*learn new weights* to produce it -- gradient descent has no pressure to specialize
weights toward a direction that's already artificially present. Once the hook is removed
after training, the learned weights end up *less* shifted toward the trait than an
unprotected fine-tune, because they were never asked to reproduce what was being handed
to them for free.

Reuses `persona_vectors_6.ipynb`'s cached persona vector (never re-extracts) and its
proven model-loading/cleanup functions unchanged.

**Model**: Qwen/Qwen2.5-7B-Instruct

In [ ]:
import os

# Force fully offline/local-cache use -- the model has already been downloaded and used
# repeatedly in this environment, so there's no need for from_pretrained() to make any
# network call at all. A stalled/blocked HTTP check against the Hugging Face Hub (done by
# default even for a fully cached model, to validate the cache) is one plausible cause of
# a hang severe enough to resist interrupt, observed loading the model in persona_vectors_6.ipynb.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Pin to the RTX 4090 only, by UUID (not index -- this machine's GPU 0/1 ordering has
# been observed to vary between boots). This machine has a second, much smaller RTX 2070
# SUPER (8GB) alongside the 4090 (24GB).
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-3185d7f6-fae1-0c3e-25f3-ad3e260d30b8"

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
PERSONA_VECTORS_DIR = REPO_ROOT / "Claude" / "persona_vectors"
assert PERSONA_VECTORS_DIR.exists(), f"Expected cloned repo at {PERSONA_VECTORS_DIR}"
sys.path.insert(0, str(PERSONA_VECTORS_DIR))

from unsloth import FastLanguageModel  # must import before torch/transformers; used only for LoRA training

import gc
import json
import random
import time
from functools import partial

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from sft import sft_train
from validate import TrainingConfig

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print("Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.")

In [ ]:
PERSONA_VECTOR_STATE_PATH = PERSONA_VECTORS_DIR / "ckpt" / "shift_prediction_demo" / "persona_vector_state.pt"
assert PERSONA_VECTOR_STATE_PATH.exists(), (
    f"No cached persona vector at {PERSONA_VECTOR_STATE_PATH}. "
    "Run persona_vectors_6.ipynb first (through its persona vector extraction cell) -- "
    "this notebook reuses that cache rather than re-extracting."
)

state = torch.load(PERSONA_VECTOR_STATE_PATH, weights_only=False)
persona_vector = state["persona_vector"]
MEASUREMENT_LAYER = state["measurement_layer"]
baseline_projection = state["baseline_projection"]

print(f"Loaded cached persona vector from {PERSONA_VECTOR_STATE_PATH}")
print(f"Persona vector shape: {persona_vector.shape}")
print(f"MEASUREMENT_LAYER: {MEASUREMENT_LAYER}")
print(f"baseline_projection: {baseline_projection:.4f}")

MISALIGNED_2_PATH = PERSONA_VECTORS_DIR / "dataset" / "evil" / "misaligned_2.jsonl"
assert MISALIGNED_2_PATH.exists(), (
    f"Missing {MISALIGNED_2_PATH} -- run persona_vectors_6.ipynb's dataset.zip "
    "extraction cell first."
)

with open(PERSONA_VECTORS_DIR / "data_generation" / "trait_data_eval" / "evil.json") as f:
    evil_eval_data = json.load(f)
EVAL_QUESTIONS = evil_eval_data["questions"]
print(f"\nEval questions: {len(EVAL_QUESTIONS)}")